## C1 - WEEK 1

- Adrià Cantarero Carreras
- Ivan Martin Campoy (1673246@uab.cat)
- Irene Pumares Benaiges (1810704@uab.cat)
- Lavish Ramchandani

In [1]:
#needed libraries

import os
import re
import cv2
import pickle
import numpy as np

## TASK 1: Compute image descriptors

In [3]:
#database and qsd1 paths

bbdd_path = "./BBDD"
qsd1_path = "./qsd1_w1"

COLOR_SPACES = {
    "GRAY": cv2.COLOR_BGR2GRAY,
    "RGB":  cv2.COLOR_BGR2RGB,
    "HSV":  cv2.COLOR_BGR2HSV,
    "LAB":  cv2.COLOR_BGR2LAB,
    "YCRCB": cv2.COLOR_BGR2YCrCb,
}

#value range of every channel. OpenCV hue goes from 0 to 179, not to 255
CHANNEL_RANGES = {
    "GRAY":  [(0, 256)],
    "RGB":   [(0, 256)] * 3,
    "HSV":   [(0, 180), (0, 256), (0, 256)],
    "LAB":   [(0, 256)] * 3,
    "YCRCB": [(0, 256)] * 3,
}

#descriptors
bbdd_descriptors = []
qsd1_descriptors = []


#----------------------------------------------------------------------------------
#sort files
def return_number(string):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', string)]

#sort bbdd files
bbdd_files = [f for f in os.listdir(bbdd_path) if f.lower().endswith(".jpg")]
bbdd_files.sort(key=return_number)

#sort qsd1 files
qsd1_files = [f for f in os.listdir(qsd1_path) if f.lower().endswith(".jpg")]
qsd1_files.sort(key=return_number)

#id of an image = number in its file name (bbdd_00007.jpg -> 7)
def image_id(file_name):
    return int(re.findall(r'\d+', file_name)[-1])

bbdd_ids = np.array([image_id(f) for f in bbdd_files])
qsd1_ids = np.array([image_id(f) for f in qsd1_files])
#----------------------------------------------------------------------------------

#feature extraction function

def compute_1d_Histogram(image, color_space = "HSV", bins = 32):
    if color_space in COLOR_SPACES:
        img_converted = cv2.cvtColor(image, COLOR_SPACES[color_space])
    else:
        img_converted = image #in case it is already in BGR

    ranges = CHANNEL_RANGES.get(color_space, [(0, 256)] * 3)

    #calc hist
    if len(img_converted.shape) == 2: #grayscale
        hist = cv2.calcHist([img_converted], [0], None, [bins], list(ranges[0]))
        descriptor = hist.flatten() #convert to 1D array


    elif len(img_converted.shape) == 3:
        channel_hists = []

        for channel in range(3):
            ch_hist = cv2.calcHist([img_converted], [channel], None, [bins], list(ranges[channel]))
            channel_hists.append(ch_hist.flatten())
        descriptor = np.concatenate(channel_hists) #stack all the channels into 1D array

    #normalization
    total_pixels = np.sum(descriptor)
    if total_pixels > 0:
        descriptor = descriptor / total_pixels

    return descriptor

#----------------------------------------------------------------------------------

#read one image
def read_image(img_path):
    image = cv2.imread(img_path)
    if image is None:
        raise IOError(f"Could not read {img_path}")   #skipping it would shift every index after it and break the ground truth
    return image

#load images iteratively bbdd set
bbdd_imgs = []
for file_name in bbdd_files:
    img_path = os.path.join(bbdd_path, file_name)
    image = read_image(img_path)
    bbdd_imgs.append(image)

##load images iteratively qsd1 set
qsd1_imgs = []
for file_name in qsd1_files:
    img_path = os.path.join(qsd1_path, file_name)
    image = read_image(img_path)
    qsd1_imgs.append(image)

In [9]:
#comparison

#METHOD 1: Grayscale
bbdd_desc_m1 = [compute_1d_Histogram(im, color_space="GRAY", bins=32) for im in bbdd_imgs]
qsd1_desc_m1 = [compute_1d_Histogram(im, color_space="GRAY", bins=32) for im in qsd1_imgs]

bbdd_desc_m1 = np.array(bbdd_desc_m1)  #shape: (287, 32)
qsd1_desc_m1 = np.array(qsd1_desc_m1)  #shape: (30, 32)

print(f" Shape of bbdd_desc_m1: {bbdd_desc_m1.shape}")
print(f" Shape of qsd1_desc_m1: {qsd1_desc_m1.shape}\n")

#METHOD 2: HSV
bbdd_desc_m2 = [compute_1d_Histogram(im, color_space="HSV", bins=32) for im in bbdd_imgs]
qsd1_desc_m2 = [compute_1d_Histogram(im, color_space="HSV", bins=32) for im in qsd1_imgs]

bbdd_desc_m2 = np.array(bbdd_desc_m2)  #shape: (287, 96)
qsd1_desc_m2 = np.array(qsd1_desc_m2)  #shape: (30, 96)

print(f" Shape of bbdd_desc_m2: {bbdd_desc_m2.shape}")
print(f" Shape of qsd1_desc_m2: {qsd1_desc_m2.shape}")


#EXTRA METHODS
from utils.extra_descriptors import *

#METHOD 3: Chroma only, LAB (a + b channels, no lightness)
bbdd_desc_m3 = [hist_chroma(im, color_space="LAB", bins=32) for im in bbdd_imgs]
qsd1_desc_m3 = [hist_chroma(im, color_space="LAB", bins=32) for im in qsd1_imgs]

bbdd_desc_m3 = np.array(bbdd_desc_m3)  #shape: (287, 64)
qsd1_desc_m3 = np.array(qsd1_desc_m3)  #shape: (30, 64)

print(f" Shape of bbdd_desc_m3: {bbdd_desc_m3.shape}")
print(f" Shape of qsd1_desc_m3: {qsd1_desc_m3.shape}\n")

#METHOD 4: Chroma only, HSV (H + S channels, no value)
bbdd_desc_m4 = [hist_chroma(im, color_space="HSV", bins=32) for im in bbdd_imgs]
qsd1_desc_m4 = [hist_chroma(im, color_space="HSV", bins=32) for im in qsd1_imgs]

bbdd_desc_m4 = np.array(bbdd_desc_m4)  #shape: (287, 64)
qsd1_desc_m4 = np.array(qsd1_desc_m4)  #shape: (30, 64)

print(f" Shape of bbdd_desc_m4: {bbdd_desc_m4.shape}")
print(f" Shape of qsd1_desc_m4: {qsd1_desc_m4.shape}\n")

#METHOD 5: CLAHE on lightness + LAB histogram
bbdd_desc_m5 = [hist_clahe_lab(im, bins=32) for im in bbdd_imgs]
qsd1_desc_m5 = [hist_clahe_lab(im, bins=32) for im in qsd1_imgs]

bbdd_desc_m5 = np.array(bbdd_desc_m5)  #shape: (287, 96)
qsd1_desc_m5 = np.array(qsd1_desc_m5)  #shape: (30, 96)

print(f" Shape of bbdd_desc_m5: {bbdd_desc_m5.shape}")
print(f" Shape of qsd1_desc_m5: {qsd1_desc_m5.shape}\n")

#METHOD 6: Saturation-weighted hue histogram
bbdd_desc_m6 = [hist_hue_weighted(im, bins=36) for im in bbdd_imgs]
qsd1_desc_m6 = [hist_hue_weighted(im, bins=36) for im in qsd1_imgs]

bbdd_desc_m6 = np.array(bbdd_desc_m6)  #shape: (287, 36)
qsd1_desc_m6 = np.array(qsd1_desc_m6)  #shape: (30, 36)

print(f" Shape of bbdd_desc_m6: {bbdd_desc_m6.shape}")
print(f" Shape of qsd1_desc_m6: {qsd1_desc_m6.shape}\n")

#METHOD 7: Gradient orientation histogram
bbdd_desc_m7 = [hist_gradient_orientation(im, bins=18) for im in bbdd_imgs]
qsd1_desc_m7 = [hist_gradient_orientation(im, bins=18) for im in qsd1_imgs]

bbdd_desc_m7 = np.array(bbdd_desc_m7)  #shape: (287, 18)
qsd1_desc_m7 = np.array(qsd1_desc_m7)  #shape: (30, 18)

print(f" Shape of bbdd_desc_m7: {bbdd_desc_m7.shape}")
print(f" Shape of qsd1_desc_m7: {qsd1_desc_m7.shape}\n")

#METHOD 8: Global LBP histogram
bbdd_desc_m8 = [hist_lbp(im) for im in bbdd_imgs]
qsd1_desc_m8 = [hist_lbp(im) for im in qsd1_imgs]

bbdd_desc_m8 = np.array(bbdd_desc_m8)  #shape: (287, 256)
qsd1_desc_m8 = np.array(qsd1_desc_m8)  #shape: (30, 256)

print(f" Shape of bbdd_desc_m8: {bbdd_desc_m8.shape}")
print(f" Shape of qsd1_desc_m8: {qsd1_desc_m8.shape}")

 Shape of bbdd_desc_m1: (287, 32)
 Shape of qsd1_desc_m1: (30, 32)

 Shape of bbdd_desc_m2: (287, 96)
 Shape of qsd1_desc_m2: (30, 96)
 Shape of bbdd_desc_m3: (287, 64)
 Shape of qsd1_desc_m3: (30, 64)

 Shape of bbdd_desc_m4: (287, 64)
 Shape of qsd1_desc_m4: (30, 64)

 Shape of bbdd_desc_m5: (287, 96)
 Shape of qsd1_desc_m5: (30, 96)

 Shape of bbdd_desc_m6: (287, 36)
 Shape of qsd1_desc_m6: (30, 36)

 Shape of bbdd_desc_m7: (287, 18)
 Shape of qsd1_desc_m7: (30, 18)

 Shape of bbdd_desc_m8: (287, 256)
 Shape of qsd1_desc_m8: (30, 256)


- - -
## Task 2: Similarity measures
- - -

In [5]:
eps = 1e-10  # small value to avoid division by zero


def euclidean_distance(x, y):
    return np.sqrt(np.sum((x - y) ** 2, axis=-1))

def L1_distance(x, y):
    return np.sum(np.abs(x - y), axis=-1)

def chi_squared_distance(x, y):
    return 0.5 * np.sum(((x - y) ** 2) / (x + y + eps), axis=-1)

def jensen_shannon_distance(x, y):
    m = 0.5 * (x + y)
    kl_x = np.sum(x * np.log((x + eps) / (m + eps)), axis=-1)
    kl_y = np.sum(y * np.log((y + eps) / (m + eps)), axis=-1)
    return np.sqrt(np.clip(0.5 * (kl_x + kl_y), 0, None))

def emd_distance(x, y, bins):
    n_hist = x.shape[-1] // bins
    cx = np.cumsum(x.reshape(x.shape[:-1] + (n_hist, bins)), axis=-1)
    cy = np.cumsum(y.reshape(y.shape[:-1] + (n_hist, bins)), axis=-1)
    return np.sum(np.abs(cx - cy), axis=(-2, -1))

#all the measures below are written as distances (smaller = more similar) because the retrieval step will rank every measure 
#with the same rule (sort ascending and take the first k)

def histogram_intersection_distance(x, y):
    return 1.0 - np.sum(np.minimum(x, y), axis=-1)

def hellinger_distance(x, y):
    return (1.0 / np.sqrt(2.0)) * np.sqrt(np.sum((np.sqrt(np.clip(x, 0, None)) - np.sqrt(np.clip(y, 0, None))) ** 2, axis=-1))

def cosine_distance(x, y):
    return 1.0 - np.sum(x * y, axis=-1) / (np.linalg.norm(x, axis=-1) * np.linalg.norm(y, axis=-1) + eps)

def correlation_distance(x, y):
    xc = x - np.mean(x, axis=-1, keepdims=True)
    yc = y - np.mean(y, axis=-1, keepdims=True)
    return 1.0 - np.sum(xc * yc, axis=-1) / (np.linalg.norm(xc, axis=-1) * np.linalg.norm(yc, axis=-1) + eps)

MEASURES = {
    "euclidean":      euclidean_distance,
    "L1":             L1_distance,
    "chi2":           chi_squared_distance,
    "intersection":   histogram_intersection_distance,
    "hellinger":      hellinger_distance,
    "cosine":         cosine_distance,
    "correlation":    correlation_distance,
    "jensen_shannon": jensen_shannon_distance,
    "emd":            emd_distance,
}

In [6]:
# Comparison between all measures on real data

query0 = qsd1_desc_m2[0]    #first query image descriptor (HSV)
database = bbdd_desc_m2     #database of 287 museum images

print(f'Query 0 evaluated against the database of {len(database)} museum images:\n')

for name, measure in MEASURES.items():
  if name == 'emd':
    dists = measure(query0, database, bins=32)
  else:
    dists = measure(query0, database)

  top5_idx = np.argsort(dists)[:5] #smallest distances = most similar
  retrieved_ids = bbdd_ids[top5_idx].tolist()

  print(f'{name.capitalize():<15} Top-5 IDs: {retrieved_ids}')

Query 0 evaluated against the database of 287 museum images:

Euclidean       Top-5 IDs: [204, 222, 120, 278, 95]
L1              Top-5 IDs: [120, 222, 204, 229, 95]
Chi2            Top-5 IDs: [120, 222, 229, 232, 95]
Intersection    Top-5 IDs: [120, 222, 204, 229, 95]
Hellinger       Top-5 IDs: [120, 222, 229, 232, 95]
Cosine          Top-5 IDs: [204, 222, 120, 278, 95]
Correlation     Top-5 IDs: [204, 222, 120, 278, 95]
Jensen_shannon  Top-5 IDs: [120, 222, 229, 232, 95]
Emd             Top-5 IDs: [120, 236, 222, 224, 182]


- - -
## Task 3: For each image in QSD1, compute similarities to museum images
- - -

In [39]:
# import ml_metrics as metrics #Couldn't install it so I copied the code from the git

import numpy as np

#code from https://github.com/benhamner/Metrics/blob/master/Python/ml_metrics/custom/kdd_average_precision.py
def kdd_apk(actual, predicted, k=10):
    """
    Computes the average precision at k for Track 1 of the 2012 KDD Cup.

    This modified version uses the number of actual clicks as the denominator,
    regardless of k.

    This function computes the average prescision at k between two lists of
    items.

    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements

    Returns
    -------
    score : double
            The average precision at k over the input lists

    """
    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        if p in actual and p not in predicted[:i]: #p not in predicted[:i] is to avoid no extra marks for repeating
            num_hits += 1.0
            score += num_hits / (i+1.0)

    if not actual:
        return 0.0

    return score / len(actual)

def kdd_mapk(actual, predicted, k=10):
    """
    Computes the mean average precision at k for Track 1 of the 2012 KDD Cup.

    This modified version uses the number of actual clicks as the denominator,
    regardless of k.

    This function computes the mean average prescision at k between two lists
    of lists of items.

    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements

    Returns
    -------
    score : double
            The mean average precision at k over the input lists

    """
    return np.mean([kdd_apk(a,p,k) for a,p in zip(actual, predicted)])


#apk is like the average precision of hits at k items
#but mapk looks in the entire dataset

In [40]:
gt_path = os.path.join(qsd1_path, "gt_corresps.pkl") #ground truth correspondences

with open(gt_path, "rb") as f:
    gt_corresps = pickle.load(f)

print("Total queries in GT:", len(gt_corresps))
print("Sample GT entries:", gt_corresps[:3])

Total queries in GT: 30
Sample GT entries: [[120], [170], [277]]
